# Mini Project: Which neighbourhood problems matter most?

Time to be the analyst. You have the tools: load, clean, filter, group, plot. This project is guided but open-ended. You will follow six steps, check the objective ones, and finish with a short written finding of your own.

**Learning outcomes**

- Plan a small analysis from question to conclusion.
- Load and clean a real dataset (dates, missing values, placeholders).
- Combine filtering and `groupby` to compare boroughs.
- Present a result as a clearly labelled chart and a short written finding.
- Reuse the same skills on a second dataset (Montreal bike counts).

**Before you start:** this notebook uses everything from `08_ReadingCSV` to `12_GroupBy`. If a step feels shaky, jump back to the notebook named in its hint.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from course_checks import check, check_equal
from course_viz.grouping import (roadmap, dashboard, chart_makeover, explore_question,
                                 progress_checklist, weekday_bars, WEEKDAYS)

Here is the road ahead. The checklist at the very end of the notebook lets you tick off each stage.

In [ ]:
roadmap()

## Step 1. Load and clean

The recipe is the same as in the last two notebooks. Read it line by line and make sure you can say what each one does (hint: `11_Dates_and_Missing_Values`).

In [ ]:
df = pd.read_csv('311-service-requests.csv', dtype={'Incident Zip': str})

FMT = '%m/%d/%Y %I:%M:%S %p'
df['Created'] = pd.to_datetime(df['Created Date'], format=FMT)
df['Closed'] = pd.to_datetime(df['Closed Date'], format=FMT)

df['hours'] = (df['Closed'] - df['Created']).dt.total_seconds() / 3600
df.loc[df['hours'] < 0, 'hours'] = np.nan          # impossible durations

df['Borough'] = df['Borough'].replace('Unspecified', np.nan)
print(df.shape)

**You should see:** `(111069, 55)`, and then two quick sanity checks below should both print a checkmark.

In [ ]:
check_equal(lambda: df['Borough'].isna().sum(), 7107, hint="Unspecified boroughs should be NaN: did the replace line run?")
check_equal(lambda: str(df['Created'].dtype).startswith('datetime64'), True, hint="Created should be a real date column")

## Step 2. Pick a question

Which neighbourhood problems matter most? "Matter" can mean *many complaints*, *slow to fix* or *a big share of a borough's complaints*. Pick one angle (or invent your own):

| Idea | Columns you need |
|---|---|
| A. Which complaint types are most common, and where? | `Complaint Type`, `Borough` |
| B. Which problems take longest to resolve? | `Complaint Type`, `hours` |
| C. Is a problem concentrated in one borough? | `Complaint Type`, `Borough` |
| D. When during the day are problems reported? | `Created`, `.dt.hour` |

**Write your question here (double-click this cell):** *My question is: ...*

The automatic checks below follow one example question, **"Is heating trouble concentrated in some boroughs?"**, so you always have a worked path. If you pick another question, do the same steps with your own complaint type, and just ignore a ❌ where the check expects heating.

Quick look to choose from:

In [ ]:
df['Complaint Type'].value_counts().head(8)

**You should see:** a list led by `HEATING` (14200), then `GENERAL CONSTRUCTION` (7471), `Street Light Condition` (7117) and `DOF Literature Request` (5797).

## Step 3. Filter

### ✏️ Your turn (filter)

Keep only the heating complaints (`Complaint Type` equal to `'HEATING'`) in a new DataFrame `topic`.

In [ ]:
topic = ...  # <- replace ... with a boolean filter on df

In [ ]:
check_equal(lambda: len(topic), 14200, hint="df[df['Complaint Type'] == 'HEATING']")

<details>
<summary>Show solution</summary>

```python
topic = df[df['Complaint Type'] == 'HEATING']
```

</details>

## Step 4. Group

Counting heating complaints per borough is not fair: big boroughs have more of everything. A better question is *what share of each borough's complaints is about heating?* That is the "mean of a True/False column" trick from `12_GroupBy`.

### ✏️ Your turn (group)

Store the share of heating complaints per borough in `heating_share` (a Series indexed by borough). Use `df`, not `topic`: you need all complaints to get a share.

In [ ]:
heating_share = ...  # <- replace ... with your groupby

In [ ]:
check_equal(lambda: heating_share.round(4).to_dict(),
            {'BRONX': 0.2079, 'BROOKLYN': 0.1383, 'MANHATTAN': 0.1439, 'QUEENS': 0.0851, 'STATEN ISLAND': 0.0351},
            hint="(df['Complaint Type'] == 'HEATING').groupby(df['Borough']).mean()")

<details>
<summary>Show solution</summary>

```python
heating_share = (df['Complaint Type'] == 'HEATING').groupby(df['Borough']).mean()
```

</details>

## Step 5. Visualize

A chart should be readable in five seconds: sorted bars, a title that says what you see, labelled axes.

### ✏️ Your turn (visualize)

Plot `share` as horizontal bars. Then add a **title** (`ax.set_title(...)`) and **x-axis label** (`ax.set_xlabel(...)`) so a stranger understands the chart. Keep the variable name `ax`.

In [ ]:
share = (df['Complaint Type'] == 'HEATING').groupby(df['Borough']).mean().sort_values() * 100

ax = share.plot(kind='barh', figsize=(7, 3))
# add ax.set_title(...) and ax.set_xlabel(...) below this line
plt.show()

In [ ]:
check(lambda: ax.get_title() != '' and ax.get_xlabel() != '', hint="Call ax.set_title('...') and ax.set_xlabel('...') before plt.show()")

<details>
<summary>Show solution</summary>

```python
share = (df['Complaint Type'] == 'HEATING').groupby(df['Borough']).mean().sort_values() * 100

ax = share.plot(kind='barh', figsize=(7, 3))
ax.set_title('Heating complaints as a share of each borough\'s complaints')
ax.set_xlabel('Percent of complaints')
ax.set_ylabel('Borough')
plt.show()
```

</details>

**You should see:** five bars, sorted from `STATEN ISLAND` (about 3.5%) at the bottom to `BRONX` (about 20.8%) at the top, with your title and label.

### Good chart, bad chart

Your rubric says "a chart readable in five seconds". Here is what that means. Both charts show the **same five numbers**.

In [ ]:
chart_makeover()

**What to notice:** the left chart is technically correct but misleading: starting the axis at 8 makes Staten Island's bar disappear and shrinks Queens to a stub, when the real ratio between the Bronx and Staten Island is about 6 to 1. The right chart fixes this and states the finding in its title. Use the right-hand chart as your standard.

## Step 5b. A second dataset: Montreal bikes

Same skills, new data. Which weekday and which month are busiest at the `Berri 1` counter?

In [ ]:
bikes = pd.read_csv('bikes.csv', sep=';', encoding='latin1',
                    parse_dates=['Date'], dayfirst=True, index_col='Date')
bikes = bikes.dropna(axis=1, how='all')
berri = bikes['Berri 1']
berri.head(3)

**You should see:** the first three days of January 2012 with counts `35`, `83`, `135`, indexed by date.

### ✏️ Your turn (bikes)

Store the **name of the busiest weekday** (by total rides) in `busiest_weekday` and the **number of the busiest month** in `busiest_month`. Hint: group `berri` by `berri.index.day_name()` and `berri.index.month`, `sum()`, then `idxmax()`.

In [ ]:
busiest_weekday = ...  # <- replace ... with your code
busiest_month = ...    # <- replace ... with your code

In [ ]:
check_equal(lambda: busiest_weekday, 'Thursday', hint="berri.groupby(berri.index.day_name()).sum().idxmax()")
check_equal(lambda: int(busiest_month), 7, hint="berri.groupby(berri.index.month).sum().idxmax()")

<details>
<summary>Show solution</summary>

```python
busiest_weekday = berri.groupby(berri.index.day_name()).sum().idxmax()
busiest_month = berri.groupby(berri.index.month).sum().idxmax()
```

</details>

## Step 6. Write your finding

Data work ends with words. In **exactly three sentences**: (1) what you found, with a number, (2) a caution about the data, (3) what you would look at next.

Example for a different question: *"Noise makes up almost 18% of Manhattan's complaints, versus under 4% in the Bronx. This is one month of data (October 2013), so a seasonal pattern may be missing. Next I would compare noise by hour of day."*

### ✏️ Your turn (finding)

Write your own three sentences about **your** question in the string `finding`. Replace the `TODO` text (the check only verifies that you wrote at least three sentences, ending with `.`).

In [ ]:
finding = 'TODO'  # <- write your three sentences here, inside the quotes

In [ ]:
check(lambda: 'TODO' not in finding and finding.count('.') >= 3, hint="Replace TODO and write three sentences, each ending in a full stop.")

<details>
<summary>Show solution</summary>

```python
finding = (
    "Heating makes up about 21% of the Bronx's complaints but only 3.5% of Staten Island's. "
    "The data covers only October 2013, the start of the heating season, so this may not hold in summer. "
    "Next I would check whether Bronx heating complaints also take longer to resolve."
)
```

</details>

## Step 5c. Put it together: a dashboard

Analysts rarely send one chart; they send a page. This dashboard re-uses the groupings you just wrote: a bar chart of heating share, the bike weekday bars, the bike monthly line and a text summary. All four panels share one palette, one title style and one-colour highlighting for the key bar.

In [ ]:
fig = dashboard(df, berri)

**What to notice:** each panel answers one question and its title says which. Orange is always "the maximum", blue is always "the rest". The text panel carries the caution from Step 6.

### ✏️ Your turn (dashboard panel)

Recreate the **bike weekday panel** yourself. The variable `weekday_mean` already holds the average per weekday in calendar order. Draw it as vertical bars on `ax` (use the first three letters as labels), then add a **title** and a **y-axis label**.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
weekday_mean = berri.groupby(berri.index.day_name()).mean().reindex(WEEKDAYS)
# draw the bars, then add ax.set_title(...) and ax.set_ylabel(...) below this line
plt.show()

In [ ]:
check(lambda: len(ax.patches) == 7 and ax.get_title() != '' and ax.get_ylabel() != '', hint="ax.bar(weekday_mean.index.str[:3], weekday_mean.to_numpy()), then ax.set_title(...) and ax.set_ylabel(...)")

<details>
<summary>Show solution</summary>

```python
fig, ax = plt.subplots(figsize=(6, 3))
weekday_mean = berri.groupby(berri.index.day_name()).mean().reindex(WEEKDAYS)
ax.bar(weekday_mean.index.str[:3], weekday_mean.to_numpy(), color='#56b4e9')
ax.set_title('Berri 1: average riders per day, by weekday')
ax.set_ylabel('Average riders per day')
plt.show()
```

</details>

### Explore your own question live

Pick a borough and a metric to see which complaint types stand out there. Use it to choose or sharpen the question for Step 6.

In [ ]:
_ = explore_question(df)

**What to notice:** switching the metric from `complaints` to `% of borough` or `median hours` can reorder the bars completely. That is why the rubric asks for shares or averages, not only counts.

## Self-assessment rubric

Tick each box honestly (double-click and change `[ ]` to `[x]`):

- [ ] My question is written down and has a clear answer in the data.
- [ ] I handled missing values or placeholders on purpose (and can say why).
- [ ] I compared groups with a *share* or an *average*, not only raw counts.
- [ ] My chart has a title, axis labels and sorted bars.
- [ ] My finding has a number, a caution and a next step.
- [ ] I can explain every line of my code to a friend.

## Stretch ideas

- **Slowest to fix:** group by `Complaint Type`, take the median `hours`, and keep types with at least 1000 complaints. Which problem waits longest?
- **Two keys:** group by `Borough` and `Agency`; which agency is busiest in each borough? (`reset_index()` helps.)
- **Time of day:** compare the hourly pattern of noise complaints in Manhattan versus Queens.
- **Bikes:** compare all seven counters. Which has the biggest weekday-versus-weekend difference?
- **Fairness:** the Bronx has the longest median wait (about 23 hours). Is that true for every complaint type, or driven by a few?

### Progress checklist

Tick what you have done; the bar fills as you go.

In [ ]:
_ = progress_checklist([
    'Loaded and cleaned the data',
    'Wrote down my question',
    'Filtered and grouped (share or average)',
    'Chart with title, axis labels, sorted bars',
    'Looked at the bike weekday and month',
    'Wrote a three-sentence finding',
])

## ✅ Check yourself

1. Why compare *shares* instead of raw counts across boroughs?
2. Why did we turn `'Unspecified'` into a missing value before grouping?
3. Which one line gives the busiest weekday in a bike series called `s` with dates as index?

<details>
<summary>Show answers</summary>

1. Boroughs differ in size, so raw counts mostly reflect population; a share compares like with like.
2. So `groupby` skips those rows instead of treating "Unspecified" as a sixth borough.
3. `s.groupby(s.index.day_name()).sum().idxmax()`

</details>

## Next

Congratulations, you finished the pandas module! You can load real data, clean it, ask questions and show the answers. Natural next steps:

- **Merge and join:** combine two tables (for example, add population per borough to compute complaints per person).
- **seaborn:** nicer statistical charts with a few lines of code.
- **pivot tables and `unstack`:** reshape results into grids like weekday x hour.
- **Your own data:** try this workflow on a CSV from your work, school or a public open-data portal.